# Competing risks: event coding, cumulative incidence, and cause-specific hazards

This notebook uses a bounded DuckDB sample from `data/processed/cases_clean.parquet`. It preserves four observed endpoints: judgment, withdrawal, transfer, and the heterogeneous `other_observed` aggregate.

The notebook does not implement Fine-Gray. Results are bounded-sample diagnostics, not full-cohort estimates.

In [ ]:
from pathlib import Path
import sys

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.competing_risks import (
    cumulative_incidence_by_group,
    encode_competing_events,
    fit_cause_specific_hazards,
)

PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cases_clean.parquet'
ROWS_PER_COMBINATION = 500
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(PROCESSED_PATH)
print({'processed_path': str(PROCESSED_PATH), 'rows_per_combination': ROWS_PER_COMBINATION})


## Bounded sample and explicit covariates

The query caps each state/event/disposal combination at 500 rows. State dummy variables are created in pandas so the Cox helper receives numeric covariates explicitly.

In [ ]:
sample_query = """
SELECT duration, event, disposal_type, state, filing_month, criminal
FROM (
    SELECT duration, event, disposal_type, state, filing_month, criminal,
           row_number() OVER (
               PARTITION BY state, event, disposal_type ORDER BY ddl_case_id
           ) AS row_number
    FROM read_parquet(?)
)
WHERE row_number <= ?
"""

with duckdb.connect() as con:
    sample = con.execute(
        sample_query, [str(PROCESSED_PATH), ROWS_PER_COMBINATION]
    ).fetchdf()

state_dummies = pd.get_dummies(sample['state'], prefix='state', drop_first=True, dtype=float)
analysis_frame = pd.concat(
    [
        sample[['duration', 'event', 'disposal_type', 'state']].reset_index(drop=True),
        state_dummies.reset_index(drop=True),
        sample[['filing_month', 'criminal']].astype(float).reset_index(drop=True),
    ],
    axis=1,
)
COVARIATES = list(state_dummies.columns) + ['filing_month', 'criminal']
print({'rows': len(analysis_frame), 'states': analysis_frame['state'].nunique()})
display(analysis_frame.groupby(['event', 'disposal_type'], dropna=False).size().rename('rows').to_frame())


## Event coding

Rows with `event=0` remain censored even if a disposal label is present. Observed labels outside the three named causes become the explicit `other_observed` endpoint.

In [ ]:
encoded = encode_competing_events(analysis_frame)
display(encoded['competing_event'].value_counts().sort_index().rename('rows').to_frame())


## Aalen-Johansen cumulative incidence

The helper uses a fixed jitter seed because integer-day durations produce tied event times in Lifelines. The resulting curves are descriptive bounded-sample estimates.

In [ ]:
cumulative_incidence = cumulative_incidence_by_group(analysis_frame, 'state')
print({
    'curve_rows': len(cumulative_incidence),
    'groups': cumulative_incidence['state'].nunique(),
    'causes': cumulative_incidence['cause'].nunique(),
})
final_curves = (
    cumulative_incidence.sort_values('duration')
    .groupby(['state', 'cause'], as_index=False)
    .tail(1)
    .sort_values(['state', 'cause'])
)
display(final_curves)


## Cause-specific Cox hazards

Each model treats its target endpoint as the event and all other endpoints as censored. The `other_observed` coefficients describe an aggregate category, not one specific disposal mechanism.

In [ ]:
cause_models = fit_cause_specific_hazards(analysis_frame, COVARIATES)
hazard_ratios = pd.DataFrame({
    cause: model.hazard_ratios_ for cause, model in cause_models.items()
})
print({'models': list(cause_models), 'covariates': COVARIATES})
display(hazard_ratios)


## Boundary

Fine-Gray is intentionally not implemented. The project first needs an explicit decision about whether the heterogeneous `other_observed` endpoint is acceptable as an aggregate cause or should be decomposed using additional source information.